# JWST_DINO — interactive training / smoke tests

Self-contained Lightning re-implementation of DINOv2 pre-training (no `dinov2` import).
Run cells below for quick interactive trials; use `sbatch/train_dev.sh` and
`sbatch/train_distribute.sh` for the real multi-GPU / multi-node runs.

Full run (matches the old `ps6_st3_distribute` config):
```bash
python trainer.py fit --config jwst_dino.yaml --trainer.devices=4
```

## 1. Smoke test — `fast_dev_run` (5 train + 5 val steps, 1 GPU)
Checks the whole pipeline wires up and losses are finite; no checkpoints written.

In [ ]:
!python trainer.py fit --config jwst_dino.yaml \
    --trainer.devices=1 --trainer.num_nodes=1 \
    --trainer.fast_dev_run=5 --data.num_workers=2 --data.batch_size=8

## 2. Short interactive run — a few tiny epochs on 1 GPU
Exercises validation + teacher-backbone export + checkpointing on a small schedule.

In [ ]:
!python trainer.py fit --config jwst_dino.yaml \
    --trainer.devices=1 --trainer.num_nodes=1 \
    --trainer.max_epochs=4 --trainer.limit_train_batches=20 \
    --trainer.check_val_every_n_epoch=2 --data.num_workers=4 --data.batch_size=32

## 3. Inspect an exported teacher backbone
The `.pth` uses the dinov2 `{"teacher": state_dict}` layout the benchmark loaders expect.

In [ ]:
import glob, torch
ckpts = sorted(glob.glob('outputs/jwst_dino_ps6_st3/version_*/eval/*/teacher_checkpoint.pth'))
print(f'{len(ckpts)} teacher checkpoints found')
if ckpts:
    sd = torch.load(ckpts[-1], map_location='cpu')['teacher']
    print(ckpts[-1])
    print('params:', sum(v.numel() for v in sd.values()))
    print('first keys:', list(sd.keys())[:5])